# 한국어 임베딩 입력 샘플 생성

목표:

- `sample/words.json`
- `sample/sentences.json`
- `sample/documents.json`

데이터셋:

- 단어: `binjang/NIKL-korean-english-dictionary`
- 문장: `klue/klue`의 `sts`
- 문서/문단: `heegyu/namuwiki-extracted`

기본 샘플 수는 아래 설정 셀에서 바꿀 수 있다.

In [1]:
# 필요한 패키지 설치
%pip install -q -U datasets tqdm pyarrow


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import random
import re
import shutil
from typing import Any, Iterable

from datasets import load_dataset
from tqdm.auto import tqdm

In [ ]:
# =========================
# 설정
# =========================

SEED = 42

SAMPLE_DIR = Path("sample")
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

WORD_SAMPLE_SIZE = 1_000
SENTENCE_SAMPLE_SIZE = 1_000
DOCUMENT_SAMPLE_SIZE = 300

# 단어 필터
MIN_WORD_LEN = 2
MAX_WORD_LEN = 12

# 문장 필터
MIN_SENTENCE_LEN = 10
MAX_SENTENCE_LEN = 200

# 문서/문단 필터
MIN_DOCUMENT_CHARS = 300
MAX_DOCUMENT_CHARS = 2_000

# streaming shuffle buffer.
# 너무 크게 잡으면 메모리를 더 쓰고, 너무 작으면 랜덤성이 약해진다.
DOCUMENT_SHUFFLE_BUFFER = 20_000


In [4]:
def clean_text(text: Any) -> str:
    """임베딩 입력으로 쓰기 좋게 공백을 정리한다."""
    text = str(text)
    text = text.replace("\u200b", " ")
    text = text.replace("\ufeff", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def contains_korean(text: str) -> bool:
    return bool(re.search(r"[가-힣]", text))


def unique_keep_order(items: Iterable[str]) -> list[str]:
    seen: set[str] = set()
    result: list[str] = []

    for item in items:
        item = clean_text(item)
        if not item:
            continue
        if item in seen:
            continue

        seen.add(item)
        result.append(item)

    return result


def deterministic_sample(items: Iterable[str], n: int, seed: int = SEED) -> list[str]:
    items = unique_keep_order(items)

    if len(items) < n:
        raise ValueError(f"샘플 후보가 부족함: required={n}, actual={len(items)}")

    rng = random.Random(seed)
    return rng.sample(items, n)


def extract_first_text(row: dict[str, Any], preferred_keys: list[str]) -> str | None:
    """row에서 텍스트 컬럼을 우선순위 기반으로 찾는다."""
    for key in preferred_keys:
        value = row.get(key)
        if isinstance(value, str) and value.strip():
            return value

    # fallback: 가장 긴 string 컬럼을 사용
    string_values = [
        value
        for value in row.values()
        if isinstance(value, str) and value.strip()
    ]

    if not string_values:
        return None

    return max(string_values, key=len)


def trim_document(text: str, max_chars: int = MAX_DOCUMENT_CHARS) -> str:
    """너무 긴 document를 문장 경계에 가깝게 자른다."""
    text = clean_text(text)

    if len(text) <= max_chars:
        return text

    cut = text[:max_chars].strip()

    # 마지막 문장부호 근처에서 자르기
    punctuation_positions = [
        cut.rfind("."),
        cut.rfind("!"),
        cut.rfind("?"),
        cut.rfind("다."),
        cut.rfind("요."),
    ]
    boundary = max(punctuation_positions)

    if boundary >= MIN_DOCUMENT_CHARS:
        return cut[: boundary + 1].strip()

    return cut


def write_json_array(path: Path, items: list[str]) -> None:
    path.write_text(
        json.dumps(items, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def preview_items(name: str, items: list[str], n: int = 3) -> None:
    print("=" * 80)
    print(f"{name}: {len(items)}")
    for i, item in enumerate(items[:n]):
        print(f"[{i}] {item[:500]}")

## 1. 단어 샘플 생성

`binjang/NIKL-korean-english-dictionary`의 `Form` 컬럼을 우선 사용한다.

In [5]:
word_ds = load_dataset(
    "binjang/NIKL-korean-english-dictionary",
    split="train",
)

print(word_ds)
print("columns:", word_ds.column_names)

Generating train split: 100%|██████████| 53172/53172 [00:00<00:00, 158934.91 examples/s]

Dataset({
    features: ['Form', 'Part of Speech', 'Korean Definition', 'English Definition', 'Usages', 'Vocabulary Level', 'Semantic Category'],
    num_rows: 53172
})
columns: ['Form', 'Part of Speech', 'Korean Definition', 'English Definition', 'Usages', 'Vocabulary Level', 'Semantic Category']


In [6]:
word_candidates: list[str] = []

for row in tqdm(word_ds, desc="collect words"):
    word = extract_first_text(
        row,
        preferred_keys=[
            "Form",
            "form",
            "word",
            "Word",
            "lemma",
            "surface",
        ],
    )

    if word is None:
        continue

    word = clean_text(word)

    if not (MIN_WORD_LEN <= len(word) <= MAX_WORD_LEN):
        continue

    # 단어 샘플 목적이므로 띄어쓰기 포함 항목 제거
    if re.search(r"\s", word):
        continue

    # 한국어 없는 항목 제거
    if not contains_korean(word):
        continue

    # 문법 어미/표현처럼 보이는 항목 일부 제거
    if word.startswith("-"):
        continue

    # 너무 특수문자 중심인 항목 제거
    if not re.fullmatch(r"[가-힣A-Za-z0-9·ㆍ\-]+", word):
        continue

    word_candidates.append(word)

word_candidates = unique_keep_order(word_candidates)
print("word candidates:", len(word_candidates))

words = deterministic_sample(word_candidates, WORD_SAMPLE_SIZE, seed=SEED)
preview_items("words", words, n=10)

collect words: 100%|██████████| 53172/53172 [00:01<00:00, 45017.48it/s]


word candidates: 42693
words: 1000
[0] 회사원
[1] 살해범
[2] 수캐
[3] 퍼덕이다
[4] 마그마
[5] 동류
[6] 인사이동
[7] 사고
[8] 처먹다
[9] 분전하다


## 2. 문장 샘플 생성

`klue/klue`의 `sts` config에서 `sentence1`, `sentence2`를 사용한다.

In [7]:
def load_klue_sts():
    try:
        return load_dataset("klue/klue", "sts")
    except Exception as first_error:
        print("load_dataset('klue/klue', 'sts') failed, fallback to load_dataset('klue', 'sts')")
        print("first error:", repr(first_error))
        return load_dataset("klue", "sts")


sts_ds = load_klue_sts()
print(sts_ds)

for split_name, split_ds in sts_ds.items():
    print(split_name, split_ds)
    print("columns:", split_ds.column_names)

Generating validation split: 100%|██████████| 519/519 [00:00<00:00, 263604.24 examples/s]

DatasetDict({
    train: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
        num_rows: 11668
    })
    validation: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
        num_rows: 519
    })
})
train Dataset({
    features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
    num_rows: 11668
})
columns: ['guid', 'source', 'sentence1', 'sentence2', 'labels']
validation Dataset({
    features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
    num_rows: 519
})
columns: ['guid', 'source', 'sentence1', 'sentence2', 'labels']


In [8]:
sentence_candidates: list[str] = []

# train/validation/test 중 존재하는 split 전부 사용
for split_name, split_ds in sts_ds.items():
    print("collect split:", split_name)

    for row in tqdm(split_ds, desc=f"collect sentences:{split_name}"):
        for key in ["sentence1", "sentence2"]:
            sentence = row.get(key)

            if sentence is None:
                continue

            sentence = clean_text(sentence)

            if not (MIN_SENTENCE_LEN <= len(sentence) <= MAX_SENTENCE_LEN):
                continue

            if not contains_korean(sentence):
                continue

            sentence_candidates.append(sentence)

sentence_candidates = unique_keep_order(sentence_candidates)
print("sentence candidates:", len(sentence_candidates))

sentences = deterministic_sample(sentence_candidates, SENTENCE_SAMPLE_SIZE, seed=SEED)
preview_items("sentences", sentences, n=10)

collect split: train


collect sentences:train: 100%|██████████| 11668/11668 [00:00<00:00, 38136.93it/s]


collect split: validation


collect sentences:validation: 100%|██████████| 519/519 [00:00<00:00, 39721.25it/s]

sentence candidates: 24288
sentences: 1000
[0] 매일 아침 통통한 다람쥐가 찾아옵니다.
[1] 두오모와 우피치도 걸어갈수 있고, 기차역에서는 십오분 도보 거리입니다.
[2] 자네가 지금 열어서 확인하고 싶은게 일반 냉장고야 아님 김치 냉장고야?
[3] 결국, 배달 앱 회사들은 이익을 내고 소비자와 소기업 소유주들은 피해에 대한 보상을 하고 있습니다.
[4] 이번 년도에는 어느 도시에서 단풍이 처음 피나요?
[5] 비옷과 우산 중 당신이 원하는 것을 말해주십시오.
[6] 엘레베이터가 있는 점과 야외 계단을 이용할 수 있다는 점이 가장 맘에 들었습니다.
[7] 주인분만 굉장히 친절하시고 유쾌하세요 !
[8] 저는 개인 욕실이 있는 개인 방에 머물렀습니다.
[9] 조정대상지역에 2주택을 보유하던 상태에서 지난 9월 1일 주택 1호가 조정대상지역에서 해제된 경우 일반세율을 적용받나요?


## 3. 문서/문단 샘플 생성

`heegyu/namuwiki-extracted`를 streaming 모드로 읽어서 필요한 개수만 수집한다.

데이터셋이 크므로 전체를 먼저 다운로드하지 않고, shuffle buffer를 사용해서 샘플링한다.

In [9]:
doc_ds = load_dataset(
    "heegyu/namuwiki-extracted",
    split="train",
    streaming=True,
)

print(doc_ds)

try:
    doc_iterable = doc_ds.shuffle(seed=SEED, buffer_size=DOCUMENT_SHUFFLE_BUFFER)
except Exception as error:
    print("streaming shuffle failed. fallback to sequential streaming.")
    print("error:", repr(error))
    doc_iterable = doc_ds

IterableDataset({
    features: ['title', 'text', 'contributors', 'namespace'],
    num_shards: 1
})


In [10]:
document_candidates: list[str] = []
seen_documents: set[str] = set()

# 너무 많이 훑지 않도록 안전 상한.
# 후보가 잘 안 모이면 이 값을 늘리면 된다.
MAX_SCANNED_ROWS = 200_000

for scanned, row in enumerate(tqdm(doc_iterable, desc="collect documents", total=MAX_SCANNED_ROWS), start=1):
    if scanned > MAX_SCANNED_ROWS:
        break

    raw_text = extract_first_text(
        row,
        preferred_keys=[
            "text",
            "content",
            "document",
            "article",
            "body",
            "paragraph",
        ],
    )

    if raw_text is None:
        continue

    document = clean_text(raw_text)

    if not contains_korean(document):
        continue

    if len(document) < MIN_DOCUMENT_CHARS:
        continue

    document = trim_document(document, max_chars=MAX_DOCUMENT_CHARS)

    if len(document) < MIN_DOCUMENT_CHARS:
        continue

    # 너무 짧은 문장만 나열된 항목 방지용 간단 필터
    if len(re.findall(r"[.!?]|다\.|요\.", document)) < 2:
        continue

    if document in seen_documents:
        continue

    seen_documents.add(document)
    document_candidates.append(document)

    if len(document_candidates) >= DOCUMENT_SAMPLE_SIZE:
        break

print("document candidates:", len(document_candidates))

if len(document_candidates) < DOCUMENT_SAMPLE_SIZE:
    raise ValueError(
        f"document 후보가 부족함: required={DOCUMENT_SAMPLE_SIZE}, actual={len(document_candidates)}. "
        "MAX_SCANNED_ROWS 또는 DOCUMENT_SHUFFLE_BUFFER를 늘려보세요."
    )

# streaming shuffle에서 이미 어느 정도 섞였지만, 최종적으로 한 번 더 deterministic shuffle
rng = random.Random(SEED)
rng.shuffle(document_candidates)
documents = document_candidates[:DOCUMENT_SAMPLE_SIZE]

preview_items("documents", documents, n=3)

collect documents:   0%|          | 315/200000 [00:04<51:12, 64.99it/s]  

document candidates: 300
documents: 300
[0] BEMANI 시리즈 수록곡. 작곡은 부타펀치. 내가 바로 바다의 왕자. 백전연마의 사나운 놈들을 거느리고 금은보화 빼앗으러 가자. 요호~요호~. 곡 목록으로 돌아가기 팝픈뮤직 13 카니발에 수록된 부타펀치의 곡. wac의 후일담에 의하면 당시 이 곡이 처음 구상되었을 때는 바이킹과 음식에 대한 곡이었으며, 이것은 후에 팝픈뮤직 16의 수록곡 バイキングマン로 재현되게 된다. 여담이지만, 부타펀치 명의임에도 불구하고 곡 자체는 오히려 히노데155 명의와 BPO 명의를 합한 스타일에 가깝다(...) EX채보는 그야말로 47이 무엇인지를 보여주는 47의 정석. 초반에 잠깐 48급의 동시치기 지대가 지나고 나면 그 이후는 47에 딱 적당한 수준의 종합력을 요구하는 패턴들이다. 다만 최후반의 계단 발광 후살은 주의. 46의 シャムシールの舞과 같이 47 러너라면 반드시 해봐야 할 보면. 39의 Dormir 삼대장, 42의 시즈쿠, 취소선의 이유는 일단 종합성으론 반드시 해봐야 되는 건 맞는데, 
[1] 1년의 233번째(윤년의 경우 234번째) 날에 해당한다. 1919년 - 원불교 법인절. 1942년 - 스탈린그라드 전투가 시작되었다. 1955년 - 홉킨스빌 고블린 외계인 사건 1963년 - 아에로플로트의 투폴레프 Tu-124기가 기관 고장으로 레닌그라드(現 상트페테르부르크)의 네바강에 불시착했다. 1983년 - 필리핀 상원의원 베니그노 아키노가 마닐라 국제공항에서 저격, 피살되었다. 2016년 - 2016 리우데자네이루 올림픽 폐막일 브라질 현지 시간 기준. 우리나라 시간으로는 8월 22일이었다. 2017년 - 배우 송선미 남편 피살 사건 성 구니포르토 성 룩소리오 성 막시미아노 성녀 바사 성 보노소 성 비오 10세 성 시도니오 아폴리나리스 성 아가피오 성 아나스타시오 성 아브라함 성 아비토 성 에우프레피오 성녀 치리아카 성 치셀로 성 카메리노 성 콰드라토 성 테오고니오 성 파테르노 성 프리바

## 4. JSON 저장

요청한 출력:

- `sample/words.json`
- `sample/sentences.json`
- `sample/documents.json`


In [11]:
write_json_array(SAMPLE_DIR / "words.json", words)
write_json_array(SAMPLE_DIR / "sentences.json", sentences)
write_json_array(SAMPLE_DIR / "documents.json", documents)

metadata = {
    "seed": SEED,
    "sample_sizes": {
        "words": len(words),
        "sentences": len(sentences),
        "documents": len(documents),
    },
    "datasets": {
        "words": {
            "name": "binjang/NIKL-korean-english-dictionary",
            "split": "train",
            "source_column_preferred": "Form",
        },
        "sentences": {
            "name": "klue/klue",
            "config": "sts",
            "splits": list(sts_ds.keys()),
            "source_columns": ["sentence1", "sentence2"],
        },
        "documents": {
            "name": "heegyu/namuwiki-extracted",
            "split": "train",
            "streaming": True,
            "source_column_preferred": "text",
        },
    },
    "filters": {
        "word_length": [MIN_WORD_LEN, MAX_WORD_LEN],
        "sentence_length": [MIN_SENTENCE_LEN, MAX_SENTENCE_LEN],
        "document_chars": [MIN_DOCUMENT_CHARS, MAX_DOCUMENT_CHARS],
    },
}

(SAMPLE_DIR / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("saved files:")
for path in sorted(SAMPLE_DIR.glob("*.json")):
    print("-", path, f"{path.stat().st_size / 1024:.1f} KiB")

saved files:
- sample/documents.json 1056.5 KiB
- sample/metadata.json 0.8 KiB
- sample/sentences.json 89.4 KiB
- sample/sentenses.json 89.4 KiB
- sample/words.json 15.0 KiB


## 5. ZIP으로 묶기

로컬 Jupyter라면 `sample/` 폴더를 그대로 쓰면 된다.  
Colab이라면 아래 셀에서 `sample.zip`을 다운로드할 수 있다.

In [12]:
# zip_base = Path("sample")
# zip_path = shutil.make_archive(str(zip_base), "zip", SAMPLE_DIR)

# print("created:", zip_path)

created: /home/k123s456h/embedding-quant/sample.zip


In [13]:
# Colab에서 실행 중이면 브라우저로 다운로드
try:
    from google.colab import files  # type: ignore

    files.download("sample.zip")
except Exception as error:
    print("Colab download skipped.")
    print("Jupyter/local 환경이면 sample/ 폴더 또는 sample.zip 파일을 직접 사용하세요.")
    print("reason:", repr(error))

Colab download skipped.
Jupyter/local 환경이면 sample/ 폴더 또는 sample.zip 파일을 직접 사용하세요.
reason: ModuleNotFoundError("No module named 'google'")


## 6. 이후 임베딩 생성 코드에서 사용하는 방법

```python
import json
from pathlib import Path

inputs = {
    "words": json.loads(Path("sample/words.json").read_text(encoding="utf-8")),
    "sentences": json.loads(Path("sample/sentences.json").read_text(encoding="utf-8")),
    "documents": json.loads(Path("sample/documents.json").read_text(encoding="utf-8")),
}
```